# 2. LNDL, the language in the turn

The model does not call tools through an API. It writes prose, and inside the prose three
constructs: `<lvar alias>value</lvar>`, `<lact alias>name(args)</lact>`, `OUT{...}`. `parse`
turns a turn into a `Program`; this notebook is a tour of what it accepts, what it repairs, and
what it refuses.

The grammar took its present shape after a SWE-bench run on which 52 of 952 turns (5.5%) could not
be read, one round lost each, and the shapes were predictable: a model trained on JSON function
calling writes strings the JSON way, so a Python literal inside a tag fights that training on every
value with a line break or a quote in it. Heredoc arguments and named repairs took the live rate to
0.9% (ADR-0004). It is stored with its outputs cleared; run it to see what each cell prints.

In [ ]:
from hub.backend.markup import normalize, tool_calls_text
from lionagi import LNDLError, parse
from lionagi.lndl import guidance, parse_call_noting, stray_tags

## The three constructs

In [ ]:
program = parse("""
Let me read the file, then search it.
<lvar pat>def </lvar>
<lact f1>read_file(path="src/x.py", limit=200)</lact>
""")
program

Arguments are Python literals. A `*alias` is a pointer: it is resolved at dispatch time to
whatever the alias names, so a value is never pasted back into a turn.

In [ ]:
program = parse('<lact g1>grep(pattern=*pat, text=*f1, flags=["i", "m"], limit=None)</lact>')
program.lacts[0].args

## A value with lines or quotes in it: the heredoc

A string argument may be written as a heredoc: `field=<<TAG`, the lines verbatim, then `TAG` alone
on a line. Nothing between the two lines is read, so a script, a shell pipeline or a code block
needs no escaping. The closing line may be followed by a comma, a closing parenthesis, the next
argument, or nothing; a heredoc opened and never closed runs to the end of the call. The guidance
names this as the first form for any multi-line value. Below, the next argument follows the closing
line with no comma; the comma is supplied and the repair is named in `problems`.

In [ ]:
program = parse("""<lact r1>run(cmd=<<EOF
cat > /tmp/t.py <<'PY'
print("a", 'b', {"k": [1, 2]})  # OUT{x: 1} in here is just text
PY
python /tmp/t.py
EOF
timeout=60)</lact>""")
la = program.lacts[0]
print(la.name, la.args["timeout"])
print(la.args["cmd"])
print(program.out, program.problems)

## Finishing

`OUT{name: ...}` ends the run. The value is a list of aliases that fill the named output, a single
alias holding the whole value, or a literal. Aliases in a list can each name a field through the
`spec.field alias` header form.

In [ ]:
print(parse("OUT{report: [s1, l1, w3]}").out)
print(parse("OUT{report: r9}").out)
print(parse('OUT{report: {summary: "short", longest: "poem.txt", words: 12}}').out)
print(parse("OUT{answer: 42}").out)

In [ ]:
lv = parse("<lvar report.summary s1>Three files; the poem is longest.</lvar>").lvars[0]
la = parse("<lact report.words w3>word_count(text=*r1)</lact>").lacts[0]
print(lv.spec, lv.field, lv.alias)
print(la.spec, la.field, la.alias, la.name)

## What the parser repairs, and says so

Models slip in predictable ways. The parser tries readings in a fixed order: as written, then line
breaks inside strings taken as the string meant, then the call's missing closers supplied, then a
missing comma supplied; the first that parses wins. The newline reading is silent, since the string
is what was meant. The other two are repairs, and each is named: the runtime carries the note into
the next notification with "if that is not the call you meant, write it again", so a wrong reading
costs a rewrite, not a silent wrong call. On a replay of the 55 recorded unreadable turns, 50 read.

In [ ]:
for body in [
    'read_file(path="a"',  # the closing tag stood in for the closers
    'edit(path="a.py", old="x = 1"\nnew="x = 2")',  # the comma left out, across a line break
    # real line breaks in a string
    'edit(path="a.py", old="def f():\n    return 1\n", new="def f():\n    return 2\n")',
]:
    name, args, repairs = parse_call_noting(body)
    print(f"{name} {args}\n  repairs: {', '.join(repairs) or 'none'}\n")

A `<lact>` whose body is not a call is a value the model wanted to name, usually a shell line it
meant to run through a pointer. It is kept as an `<lvar>` under its alias and the turn is told.

In [ ]:
program = parse("<lact s3>git log --oneline -5; git status</lact>\n<lact a1>run(cmd=*s3)</lact>")
print(program.lvars)
print(program.lacts)
print(program.problems[0])

## Native tool-call markup

Some models answer in their own tool-call markup even when no tool is offered. `parse` knows no
provider, so the backend rebuilds that markup before the parser reads the turn (ADR-0004/D2); the
OpenRouter backend does it with the two functions in `hub.backend.markup`. Markup the provider
left in the text is normalized: DeepSeek's `invoke` wrapper around an LNDL tag is unwrapped, and a
whole native call becomes a `<lact>` under a generated alias. Markup the provider parsed out of the
text arrives as `tool_calls` on the response and is rebuilt into the text the model wrote with
`tool_calls_text`, so the loop reads the turn instead of failing it. Many unreadable turns took this shape.

In [ ]:
wrapped = (
    "<｜DSML｜ calls>\n"
    '<｜DSML｜ invoke name="lact g1>grep(pattern="def f", path="a.py")</lact>\n'
    "</｜DSML｜ calls>"
)
print([(a.alias, a.name, a.args) for a in parse(normalize(wrapped)).lacts])

whole = """
<｜DSML｜function_calls>
<｜DSML｜invoke name="read_file">
<｜DSML｜parameter name="path">src/x.py</｜DSML｜parameter>
</｜DSML｜invoke>
</｜DSML｜function_calls>
"""
print(normalize(whole).strip())

calls = [{"type": "function", "function": {"name": 'lact a1>list_dir(path=".")</lact>', "arguments": "{}"}}]
print(tool_calls_text(calls))

`true`, `false` and `null` are read as Python literals, since the model's training data is mostly
JSON.

In [ ]:
parse('<lact a1>search(query="x", exact=true, limit=null)</lact>').lacts[0].args

Two constructs with one alias in the same turn: the first is kept, the second is dropped and
named in `problems`. Aliases are unique for the whole run, not only the turn; the runtime refuses
a reused one and says which entry already carries it.

In [ ]:
program = parse('<lact a1>read_file(path="a")</lact>\n<lact a1>read_file(path="b")</lact>')
print(len(program.lacts), program.problems)

A tag that is not one of the three is not a construct. `stray_tags` names such tags when they
happen to be command names, so the runtime can say `write <lact a1>list_dir(...)</lact>`.

In [ ]:
turn = "I will now <list_dir> the directory </list_dir> and see."
print(parse(turn).lacts)
print(stray_tags(turn, {"list_dir", "read_file"}))

## What it refuses

Not every slip has a repair. An unreadable turn is an `LNDLError`; the runtime does not end the run
on it: the error goes into the next notification, verbatim, and the model tries again. The same
error twice in a row is counted, and where the heredoc is the fix the message says so.

In [ ]:
for bad in [
    "<lvar 1x>value</lvar>",
    "<lact a1>f(1)</lact>",
    '<lact r>run(cmd="ls", timeout=)</lact>',
    "<lact r>run(cmd=python -c\nimport x)</lact>",
]:
    try:
        parse(bad)
    except LNDLError as e:
        print(f"{bad!r}\n  -> {e}\n")

An unreadable `OUT{}` is the one error that does not fail the turn. The commands in the turn
still run, `out` is `None`, and the reason lands in `problems`, so the model finishes next round
with its results in view instead of losing the work it asked for.

In [ ]:
program = parse('<lact w1>write_file(path="a.txt", text="x")</lact>\nOUT{report}')
print(program.out, [la.alias for la in program.lacts])
print(program.problems[0])

## The guidance

This is the text the model is taught the language with. It is generated from the run's
outputs so the example names the real thing, and it names the heredoc as the first form for a
multi-line value.

In [ ]:
print(guidance(("report",)))